---
## Task 4 — Finishing vs. Expected Goals (xG)

### Analytic question formulation
On average, do teams score a significantly different number of **actual goals** than their
**Expected Goals (xG)** would predict — i.e., is there over- or under-performance relative to xG
across the tournament?

### Data wrangling
Fetch the `TeamMatch` sheet from the clean dataset and compute `Diff = GoalsFor − xG` for each
team-match record. A positive value means a team scored more than their chance quality
suggested (clinical finishing); negative means under-performance.


In [1]:
import pandas as pd
import numpy as np
import re
from scipy import stats
CLEAN_PATH = "../World_Cup_2026_clean.xlsx"
team_match_t4 = pd.read_excel(CLEAN_PATH, sheet_name="TeamMatch")
team_match_t4["Diff"] = team_match_t4["GoalsFor"] - team_match_t4["xG"]
pop_diff = team_match_t4.dropna(subset=["Diff"])["Diff"]

print("Population size (team-match records):", len(pop_diff))
print("Population mean (Goals - xG):", round(pop_diff.mean(), 3))

Population size (team-match records): 206
Population mean (Goals - xG): 0.154


### Data preparation and sampling
**Population:** all 206 team-match records with both a goals and an xG value.
**Sample:** a **simple random sample of n = 35** records, drawn without replacement.


In [2]:
sample_diff = pop_diff.sample(n=35, random_state=21)
print("Sample size:", len(sample_diff))

Sample size: 35


### Descriptive statistics

In [3]:
print(sample_diff.describe())
print("Skewness:", round(stats.skew(sample_diff), 3))

count    35.000000
mean     -0.082857
std       0.813983
min      -1.580000
25%      -0.515000
50%      -0.220000
75%       0.480000
max       1.530000
Name: Diff, dtype: float64
Skewness: 0.218


### Inferential statistics — Confidence interval
95% CI for the true mean (Goals − xG).

In [4]:
mean = sample_diff.mean()
sem = stats.sem(sample_diff)
ci = stats.t.interval(0.95, df=len(sample_diff) - 1, loc=mean, scale=sem)
print(f"Sample mean diff: {mean:.3f}, SEM: {sem:.3f}")
print(f"95% CI for mean (Goals - xG): ({ci[0]:.3f}, {ci[1]:.3f})")

Sample mean diff: -0.083, SEM: 0.138
95% CI for mean (Goals - xG): (-0.362, 0.197)


### Inferential statistics — One-sample t-Test
H₀: μ(Goals − xG) = 0  vs.  H₁: μ(Goals − xG) ≠ 0


In [5]:
t_stat, p_val = stats.ttest_1samp(sample_diff, popmean=0)
print(f"t-statistic = {t_stat:.3f}, p-value = {p_val:.4f}")
alpha = 0.05
print("Conclusion:", "Reject H0" if p_val < alpha else "Fail to reject H0",
      "at the 5% significance level.")

t-statistic = -0.602, p-value = 0.5510
Conclusion: Fail to reject H0 at the 5% significance level.


**Interpretation:** p = 0.551 (> 0.05), so we fail to reject H₀ — this sample gives no
significant evidence that teams over- or under-performed their xG overall; the 95% CI (−0.36,
0.20) comfortably straddles zero, meaning actual finishing was broadly in line with underlying
chance quality across the tournament.


---
## Summary

| Task | Focal point | Test type | Result (α = 0.05) |
|---|---|---|---|
| 1 | Total goals per match vs. historical benchmark (2.5) | One-sample t-test | Not significant (p = 0.729) |
| 2 | Shots on target — Group vs Knockout stage | Two-sample t-test | Not significant (p = 0.205) |
| 3 | Corners — Winning vs Losing teams | Two-sample t-test | Not significant (p = 0.162) |
| 4 | Goals vs. Expected Goals (xG) | One-sample t-test | Not significant (p = 0.551) |

*Note:* Results depend on the specific random sample drawn (seeds are fixed here for
reproducibility). Changing `random_state` or the sample size `n` will shift the exact
statistics, though the qualitative conclusions are fairly stable given the effect sizes involved.

**File dependencies for this notebook:**
- `World_Cup_2026.xlsx` — the original raw workbook (only needed for Section 0)
- `World_Cup_2026_clean.xlsx` — produced by Section 0, then fetched independently by every task
